# Cleartext Non-DP Training

Plaintext baseline: Non-DP training without cryptographic protections.

In [ ]:
import os
import sys
import torch
from time import time
import matplotlib.pyplot as plt
import numpy as np
import crypten
import torch.nn as nn
from torch.nn.functional import cross_entropy

import nest_asyncio
nest_asyncio.apply()


device = "cuda" if torch.cuda.is_available() else "cpu"
# If on Apple Silicon, use MPS
# device = "mps"
# os.environ['PFL_PYTORCH_DEVICE'] = device

sys.path.append('./dataset/')
from dataset.fashion_mnist.load_preprocess import load_and_preprocess_fashion_mnist
from dataset.mnist.load_preprocess import load_and_preprocess_mnist

sys.path.append('./utils/')
from utils.models import ThreeLayerNN
from utils.mpc_dpsgd_trainer import DP_Trainer


## Model
The model used for the experiments is a 3 layers neural network.

In [ ]:
model = ThreeLayerNN(
    input_size=784,
	hidden_size=100,
	output_size=10
)

## Dataset
The experiments can be run on either MNIST or Fashion-MNIST datasets by changing the `dataset` variable below.

In [ ]:
dataset_name = "mnist" 
# dataset_name = "fashion_mnist"

### Pre-processing
For each dataset, we perform standard pre-processing, i.e., scaling and normalization. More details in `dataset/mnist/load_preprocess_mnist.py` and `dataset/fashion_mnist/load_preprocess_fashion_mnist.py`.

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)

crypten.init()


if dataset_name == "mnist":
	train_data, val_data = load_and_preprocess_mnist(scaling=True, normalization=True)
elif dataset_name == "fashion_mnist":
	train_data, val_data = load_and_preprocess_fashion_mnist(scaling=True, normalization=True)
else:
	raise ValueError(f"Unsupported dataset name: {dataset_name}")

x_train, y_train = train_data.data, train_data.targets
x_val, y_val = val_data.data, val_data.targets

y_train = y_train.squeeze()
y_val = y_val.squeeze()


print(f"Max value of x_train: {x_train.max()}, Min value of x_train: {x_train.min()}")
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)

## Training
To perform the non-DP training, we use our custom trainer `DP_Trainer` from `utils/mpc_dpsgd_trainer.py` which can also perform non-DP training with the `train_non_dp` method. This trainer is also used for MPC training with CrypTen.

In [ ]:
batch_size = 32
learning_rate = 0.01
epochs = 30
dataset_size = len(x_train)

dp_trainer = DP_Trainer(
    model=model,
	batch_size=batch_size,
    lr = learning_rate,
	num_epochs=epochs,
    verbose = False,
	device=device,
    num_labels=10
)

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)

dp_trainer.train_non_dp(
	x = x_train, 
	y = y_train,
	x_val = x_val,
	y_val = y_val,
	validation_freq=1
)